In [7]:
import pandas 
from pathlib import Path
import glob
import asyncio
import etl_books_async
import gitsource




# Question 1:

Get the lines of  'Think Python'. We build a ETL process to download and transform the pdfs into md. Results are stored int `books_text/`. 

#### Two-stage pipeline

- **Stage 1:** Concurrent PDF downloads *(I/O-bound)*
- **Stage 2:** Parallel PDF → Markdown conversion *(CPU-bound)*

---

#### Data source

- Reads `books.csv` from a remote URL
- Extracts PDF links from `pdf_url` *(fallback: `url`)*

---

#### Deterministic outputs

- PDFs saved in `pdfs/`
- Markdown files saved in `books_text/`
- Filenames derived from path in the folder

---

#### Concurrency model

- `asyncio` + `aiohttp` for high-throughput downloads
- `asyncio.Semaphore` to limit concurrent downloads
- `ProcessPoolExecutor` for safe parallel PDF conversion

---

#### Robustness

- Retry logic with exponential backoff for downloads
- Atomic file writes using temporary `.part` files
- Skips already-downloaded PDFs
- Skips already-converted Markdown files *(unless forced)*

---

#### Fault tolerance

- Uses `asyncio.gather(..., return_exceptions=True)`
- Continues processing even if some files fail
- Reports success/failure counts per stage

---

#### Performance-aware

- Separates I/O-bound and CPU-bound workloads
- Concurrency levels configurable:
  - `DOWNLOAD_CONCURRENCY`
  - `CONVERT_CONCURRENCY`
- Measures total execution time



In [ ]:
# Run the etl 

await etl_books_async.run_etl_async(force_md=False)


In [8]:
from pathlib import Path


md_dir = Path("books_text")

for md_file in md_dir.glob("thinkpython*.md"):
    print(f"The python book is: {md_file.name}")

    lines = md_file.read_text(encoding="utf-8").splitlines()
    content = "\n".join(lines)

    print(f"Content length (chars): {len(content)}")

    lines_cleaned = [line for line in lines if line.strip()]

    print(lines[:5])
    print(lines_cleaned[:5])

    print(".........")
    print(".........")
    
    print(
        f"There are {len(lines)} lines. "
        f"If we remove empty lines, there are {len(lines_cleaned)} lines."
    )
    
    print(".........")
    print(".........")

The python book is: thinkpython2.md
Content length (chars): 480870
['## Think Python', '', '#### How to Think Like a Computer Scientist', '', '2nd Edition, Version 2.4.0']
['## Think Python', '#### How to Think Like a Computer Scientist', '2nd Edition, Version 2.4.0', '## Think Python', '#### How to Think Like a Computer Scientist']
.........
.........
There are 16604 lines. If we remove empty lines, there are 9940 lines.
.........
.........


In [9]:
for  book in md_dir.glob("*.md"):
    lines = book.read_text(encoding="utf-8").splitlines()
    content = "\n".join(lines)
    print(f"{book.name}: has {len(content)} characters and {len(lines)} lines.")

Think-C.md: has 224694 characters and 8034 lines.
PhysicalModelingInMatlab4.md: has 258832 characters and 9499 lines.
thinkos.md: has 156981 characters and 4883 lines.
thinkcomplexity2.md: has 328697 characters and 10772 lines.
thinkdsp.md: has 223189 characters and 7705 lines.
thinkjava2.md: has 516325 characters and 17978 lines.
thinkpython2.md: has 480870 characters and 16604 lines.


# Question 2. Chunking for RAG

In [10]:
books_path = Path('books_text')


def prepare_document(md_file: Path) -> dict:
    # 1) Read file
    text = md_file.read_text(encoding="utf-8")

    # 2) Split into lines
    lines = text.splitlines()

    # 3) Remove empty/whitespace-only lines
    lines_cleaned = [line.strip() for line in lines if line.strip()]

    # 4) Join lines into one large string separated by \n
    content = "\n".join(lines_cleaned)

    # 5) Build dict
    return {
        "source": md_file.name,
        "content": content,
    }


def create_docs(md_dir: Path) -> list[dict]:
    return [prepare_document(p) for p in md_dir.glob("*.md")]


docs = create_docs(books_path)

books = []
for doc in docs:
    books.append(doc.get("source"))
    
print("Books processed:", books)

docs



Books processed: ['Think-C.md', 'PhysicalModelingInMatlab4.md', 'thinkos.md', 'thinkcomplexity2.md', 'thinkdsp.md', 'thinkjava2.md', 'thinkpython2.md']


[{'source': 'Think-C.md',
  'content': '## How to Think Like a Computer Scientist\n### C Version\n#### Thomas Scheffler\nbased on previous work by Allen B. Downey\n#### Version 1.10\nJune 27th, 2019\n**2**\nCopyright (C) 1999 Allen B. Downey\nCopyright (C) 2009 Thomas Scheffler\nPermission is granted to copy, distribute, transmit and adapt this work under the Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License: `[https://creativecommons.org/licenses/by-nc/4.0/](https://creativecommons.org/licenses/by-nc/4.0/)` .\nIf you are interested in distributing a commercial version of this work, please\ncontact the author(s).\nThe L [A] TEX source and code for this book is available from:\n```\nhttps://github.com/tscheffl/ThinkC\n```\n# **Contents**\n**1** **The way of the program** **1**\n1.1 What is a programming language? . . . . . . . . . . . . . . . . . 1\n1.2 What is a program? . . . . . . . . . . . . . . . . . . . . . . . . . 3\n1.3 What is debugging? . . . . . 

In [ ]:

python_book = [book for book in docs if book.get("source") == "thinkpython2.md"]


`chunk_documents()` logic -> the step parameter controls overlap. With size=1000, step=500, you get 500 characters of overlap (1000 - 500 = 500)

In [12]:
from gitsource import chunk_documents

python_chunks = chunk_documents(python_book, size = 100, step = 50, content_field_name="content")

# check
for chunk in python_chunks[:5]:
    print(chunk)
    print("-----")


    
print("Total chunks:", len(python_chunks))

print("Average chunk length (chars):", sum(len(chunk['content']) for chunk in python_chunks) / len(python_chunks))
print("Average chunk length (lines):", sum(chunk['content'].count('\n') + 1 for chunk in python_chunks) / len(python_chunks))
print("Average chunk length (words):", sum(len(chunk['content'].split()) for chunk in python_chunks) / len(python_chunks))



{'start': 0, 'content': '## Think Python\n#### How to Think Like a Computer Scientist\n2nd Edition, Version 2.4.0\n## Think Pyth', 'source': 'thinkpython2.md'}
-----
{'start': 50, 'content': 'Scientist\n2nd Edition, Version 2.4.0\n## Think Python\n#### How to Think Like a Computer Scientist\n2nd', 'source': 'thinkpython2.md'}
-----
{'start': 100, 'content': 'on\n#### How to Think Like a Computer Scientist\n2nd Edition, Version 2.4.0\n#### Allen Downey Green Te', 'source': 'thinkpython2.md'}
-----
{'start': 150, 'content': ' Edition, Version 2.4.0\n#### Allen Downey Green Tea Press\nNeedham, Massachusetts\nCopyright © 2015 Al', 'source': 'thinkpython2.md'}
-----
{'start': 200, 'content': 'a Press\nNeedham, Massachusetts\nCopyright © 2015 Allen Downey.\nGreen Tea Press\n9 Washburn Ave\nNeedham', 'source': 'thinkpython2.md'}
-----
Total chunks: 9428
Average chunk length (chars): 99.99946966482817
Average chunk length (lines): 3.108082308018668
Average chunk length (words): 18.482498939329

In [17]:
python_chunks[12]

{'start': 600,
 'content': '/creativecommons.org/licenses/by-nc/3.0/)` .\nThe original form of this book is L [A] TEX source code',
 'source': 'thinkpython2.md'}

# Question 3. Indexing with minsearch

In [13]:
from minsearch import Index
from collections import Counter


chunk_docs =  chunk_documents(docs, size=100, step=50, content_field_name="content")



chunks_per_doc = Counter(c["source"] for c in chunk_docs)

display(chunk_docs[:5])
display(chunks_per_doc)



[{'start': 0,
  'content': '## How to Think Like a Computer Scientist\n### C Version\n#### Thomas Scheffler\nbased on previous work',
  'source': 'Think-C.md'},
 {'start': 50,
  'content': 'rsion\n#### Thomas Scheffler\nbased on previous work by Allen B. Downey\n#### Version 1.10\nJune 27th, 2',
  'source': 'Think-C.md'},
 {'start': 100,
  'content': ' by Allen B. Downey\n#### Version 1.10\nJune 27th, 2019\n**2**\nCopyright (C) 1999 Allen B. Downey\nCopyr',
  'source': 'Think-C.md'},
 {'start': 150,
  'content': '019\n**2**\nCopyright (C) 1999 Allen B. Downey\nCopyright (C) 2009 Thomas Scheffler\nPermission is grant',
  'source': 'Think-C.md'},
 {'start': 200,
  'content': 'ight (C) 2009 Thomas Scheffler\nPermission is granted to copy, distribute, transmit and adapt this wo',
  'source': 'Think-C.md'}]

Counter({'thinkjava2.md': 10041,
         'thinkpython2.md': 9428,
         'thinkcomplexity2.md': 6408,
         'PhysicalModelingInMatlab4.md': 5073,
         'Think-C.md': 4370,
         'thinkdsp.md': 4367,
         'thinkos.md': 3085})

In [19]:
index = Index(text_fields=["content"], keyword_fields=["source"])

index.fit(chunk_docs)


In [20]:
# how many chunks are indexed per source document?

Counter(c["source"] for c in index.docs)


Counter({'thinkjava2.md': 10041,
         'thinkpython2.md': 9428,
         'thinkcomplexity2.md': 6408,
         'PhysicalModelingInMatlab4.md': 5073,
         'Think-C.md': 4370,
         'thinkdsp.md': 4367,
         'thinkos.md': 3085})

In [21]:
# vocabulary size and terms matrix shape
vec = index.vectorizers["content"]

print("Unique tokens in the index:", len(vec.vocabulary_))

# chunks x unique tokens
print(index.text_matrices["content"].shape)


Unique tokens in the index: 19875
(42772, 19875)


# 4.Searching and RAG

In [22]:
results = index.search("python function definition", num_results=5)

if results:
    top = results[0]
    print("Top source:", top["source"])

Top source: thinkpython2.md


In [23]:
results

[{'start': 73850,
  'content': ' Python, but it is also possible\nto add new functions. A **function definition** specifies the name ',
  'source': 'thinkpython2.md'},
 {'start': 87550,
  'content': 'a function definition. The name of the function is a\nvariable that refers to a function object.\n**he',
  'source': 'thinkpython2.md'},
 {'start': 87500,
  'content': 'contains.\n**function object:** A value created by a function definition. The name of the function is',
  'source': 'thinkpython2.md'},
 {'start': 92050,
  'content': 'g definition into it (Listing 5.1):\nListing 5.1: A function definition\n```\nfunction res = myfunc(x)\n',
  'source': 'PhysicalModelingInMatlab4.md'},
 {'start': 42050,
  'content': 'for Python 3, but I include some notes\nabout Python 2.\nThe Python **interpreter** is a program that ',
  'source': 'thinkpython2.md'}]

# 5. Full RAG

In [24]:
# Setup OpenAI API key

from openai import OpenAI

openai_client = OpenAI()

In [35]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)
    

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
        
    )
    
    print(f'This query used {response.usage.total_tokens} tokens in total. \n * {response.usage.input_tokens} input tokens \n * {response.usage.output_tokens} output tokens.')
    
    return response.output_text

    

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)


    
    return answer

In [36]:
query = "python function definition"

In [37]:
rag(query)

This query used 447 tokens in total. 
 * 326 input tokens 
 * 121 output tokens.


"In Python, a **function definition** specifies the name of the function and the set of operations it performs. The name of the function acts as a variable that refers to the function object created by the definition. This allows you to encapsulate code into reusable blocks that can be executed with the function name whenever required. \n\nHere's a simple example of a function definition in Python:\n\n```python\ndef my_function(x):\n    return x + 1\n```\n\nIn this example, `my_function` is defined to take one parameter `x` and returns `x` incremented by 1."

# 6. Structured vs Unstructured Output

In [ ]:
import json
from typing import Literal

from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(
        description="The category of the answer"
    )
    followup_questions: list[str] = Field(description="Suggested follow-up questions")



instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
""".strip()

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)



# Edit LLM fuunction to enforce RAGResponse schema
def llm_structured(
    question: str,
    model: str = "gpt-4o-mini",
    instructions_text: str = instructions,
):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)

    messages = []
    if instructions_text:
        messages.append({"role": "system",
                         "content": instructions_text})
    messages.append({"role": "user", 
                     "content": user_prompt})

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse 
    )

    print(
        "This query used "
        f"{response.usage.total_tokens} tokens in total.\n"
        f" * {response.usage.input_tokens} input tokens\n"
        f" * {response.usage.output_tokens} output tokens."
    )


    return response.output_parsed




In [ ]:

rag_resp= llm_structured(query)



This query used 782 tokens in total.
 * 510 input tokens
 * 272 output tokens.


{'answer': "In Python, a **function definition** is a block of code that performs a specific task and can be reused. It begins with the keyword `def`, followed by the function name and parentheses that may include parameters. Here’s a basic structure of a function definition:\n\n```python\n\ndef function_name(parameters):\n    # code block\n    return value  # optional\n```\n\n- **function_name**: The name you give to the function. It's a variable that refers to the function object.\n- **parameters**: Inputs to the function (optional).\n- **return**: This statement is optional; it returns a value from the function.\n\nExample Function Definition:\n\n```python\n\ndef greet(name):\n    return f'Hello, {name}!'\n```\n\nIn this example, `greet` is a function that takes `name` as an argument and returns a greeting string.",
 'found_answer': True,
 'confidence': 0.9,
 'confidence_explanation': 'The information is directly related to function definitions in Python, providing both structure an

In [31]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'Suggested follow-up questions',
   'items': {'type': 'string'},
   'title': 'Followup Questions',
   'type': 'array'}},
 'required': ['answer',
  'found_answer',
  'confidence',
  'confidence_expla